In [1]:
from random import choice
import networkx as nx
import numpy as np
import os
import pandas as pd
import seaborn as sns
from adjustText import adjust_text
import argparse
import json
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler

C:\Users\cmurua\Anaconda3\envs\Nominate_v2\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\cmurua\Anaconda3\envs\Nominate_v2\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
C:\Users\cmurua\Anaconda3\envs\Nominate_v2\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:

NUM_GPUS = 1
BS_PER_GPU = 64  # Cantidad de ejemplos por lote por GPU. En este caso, se establece en 64, 
                 # aunque podría ser 128.

# Definición del dispositivo
# device_name = '/gpu:0'  # Si se utiliza una GPU específica
device_name = '/cpu:0'  # Si se utiliza la CPU

# Configuración de los argumentos de línea de comandos
parser = argparse.ArgumentParser()
parser.add_argument("-d", "--dataset", default='Francia', required=False)
parser.add_argument("-e", "--epochs", default=20, type=int, required=False)
parser.add_argument("-m", "--min", default=10, type=int, required=False) # Cantidad minima de votaciones por usuario
parser.add_argument("-ps", "--space", default=4, type=int, required=False)  # Dimension de la capa de embedding. Para coordenadas bidimensionales debe ser 2
parser.add_argument("-s", "--subset", default=999999, type=int, required=False) # Cantidad máxima de usuarios de la base de datos
parser.add_argument("-t", "--type", default="default", type=str, required=False) # Tipo de modelo en la arquitectura. Como se definieron las funciones con los parámetros que estaban por defecto, este parámetro no se utiliza.
parser.add_argument("-f", required=False)
args = parser.parse_args()


# Asignación de los argumentos a variables correspondientes
DATASET = args.dataset
NUM_EPOCHS = args.epochs
MINIMUM_NUMBER_VOTES = args.min
NUMBER_SELECTED_USERS = args.subset
DIM_POLITICAL_SPACE = args.space 
MODEL_TYPE = args.type

## Lectura de datos
**Lectura del archivo que contiene información de votaciones.**

In [3]:
# Lectura del archivo con la información de las votaciones
path = f"../data/Dataset 3, france_participation.csv"  # Ruta del archivo de datos basada en el conjunto de datos seleccionado

# chunksize se refiere a cuántas filas por segundo pandas lee del archivo en cada iteración
data = pd.read_csv(path, chunksize=10**5)  # Lectura del archivo en bloques de 100,000 filas
data = pd.concat(data)  # Concatenación de los bloques de datos en un solo DataFrame
len(data['user_id'].unique())

1606

**Se define id único.**

In [4]:
data = data.reset_index()
data = data.rename(columns={'index':'id_original'})
data =data.drop(columns=['id'])

## Preprocesamiento
**1. Convierte la columna de fecha y hora al formato de fecha y hora**

In [5]:
# Convertir la columna "datetime" a formato datetime
data["created_at"] = pd.to_datetime(data["created_at"].str[0:19], format="%Y-%m-%d %H:%M:%S")

# Ordenar los datos por la columna "datetime"
data = data.sort_values("created_at")

**2. Remueve los valores NaN**

In [6]:
# Filtrar filas que contienen valores no nulos en las columnas "option_a" y "option_b"
data = data[(data["option_a"].notna()) & (data["option_b"].notna())].reset_index(drop=True)

# Convertir las columnas "option_a", "option_b" y "selected" a tipo entero
cols = ["option_a", "option_b", "selected"]
data[cols] = data[cols].astype(int)

**3. Elimina usuarios con menos de < THRESHOLD_MIN_USER > votos en la plataforma.**

In [7]:
# Crear un DataFrame temporal llamado df_temp con las columnas "id" y "uuid" del DataFrame data
df_temp = data[["id_original", "user_id"]]

# Agrupar el DataFrame df_temp por "uuid" y contar el número de ocurrencias de "id"
df_temp = df_temp.groupby(["user_id"]).agg({"id_original": "count"}).reset_index()

# Filtrar las filas en df_temp para incluir solo aquellos "uuid" con un número de ocurrencias mayor o igual a MINIMUM_NUMBER_VOTES
df_temp = df_temp[df_temp["id_original"] >= MINIMUM_NUMBER_VOTES]

# Extraer los "uuid" válidos como un arreglo de valores únicos
valid_uuid = df_temp["user_id"].unique()

# Filtrar el DataFrame data para incluir solo las filas con "uuid" en valid_uuid y crear una copia del DataFrame resultante
data = data[data["user_id"].isin(valid_uuid)].copy()

# Se imprime la cantidad de usuarios únicos
len(data['user_id'].unique())

1488

**Después de aplicar este filtro, se observa una disminución a 50,061 usuarios únicos.**

**4. Crea una nueva columna llamada "card_id". Formato: 1 + < id_de_propuesta > + < id_de_propuesta >.**

**Cada <proposal_id> incluye 3 dígitos.**

In [8]:
# Extraer los valores de las columnas "option_a", "option_b" y "selected" del DataFrame data y almacenarlos en la variable a
a = data[["option_a", "option_b", "selected"]].values

# Ordenar las opciones, siempre colocando el valor más bajo en la columna izquierda
data["option_a_sorted"] = np.where(a[:, 0] < a[:, 1], a[:, 0], a[:, 1])
data["option_b_sorted"] = np.where(a[:, 0] >= a[:, 1], a[:, 0], a[:, 1])

# Generar un ID de tarjeta
data["card_id"] = "1" + data["option_a_sorted"].astype(str).str.zfill(3) + data["option_b_sorted"].astype(str).str.zfill(3)
data["card_id"] = data["card_id"].astype(int)

**5. Incluye una nueva columna llamada "latest". Esta columna filtra las votaciones duplicadas de un usuario sobre un par de propuesta y selecciona la última votación registrada para dicho par.**

In [9]:
# Ordena los datos por datetime de los clics realizados en el dataset
data = data.sort_values(["id_original", "created_at"])

# Extrae las columnas "uuid", "card_id", "datetime" y "id" del DataFrame data y las almacena en la variable u
u = data[["user_id", "card_id", "created_at", "id_original"]]

# Agrupa los datos por "datetime", "uuid", "card_id" y "id" y cuenta las ocurrencias, luego reinicia los índices
u = u.groupby(["created_at", "user_id", "card_id", "id_original"]).count().reset_index()

# Elimina duplicados de "uuid" y "card_id" manteniendo solo las últimas ocurrencias, y elimina las columnas "datetime", "uuid" y "card_id"
u = u.drop_duplicates(["user_id", "card_id"], keep="last").drop(columns=["created_at", "user_id", "card_id"])

# Agrega una columna "latest" con valor 1 a los datos del DataFrame data
u["latest"] = 1

# Une los datos originales del DataFrame data con los datos en el DataFrame u, utilizando la columna "id" como clave y manteniendo todos los registros de data
data = pd.merge(data, u, on="id_original", how="left")

# Reemplaza los valores nulos de la columna "latest" con 0 en el DataFrame data
data.loc[data["latest"].isna(), "latest"] = 0

# Convierte la columna "latest" en tipo entero en el DataFrame data
data["latest"] = data["latest"].astype(int)

# Selecciona solo las últimas votaciones de todos los usuarios, es decir, donde la columna "latest" es igual a 1, y crea una copia del DataFrame data
data = data[data["latest"] == 1].copy()

**6. Se filtran las votaciones que no tuvieron preferencia**

In [10]:
# Se crea una copia del DataFrame dfRaw
df = data[data.columns]

# Se filtran los datos en df donde la columna 'selected' no es igual a 0
df = df[df['selected'] != 0]

# Se filtran los datos en df donde la columna 'uuid' no es nula
df = df[~df['user_id'].isna()]

# Se printea la cantidad de usuarios únicos
len(df['user_id'].unique())

1488

**Con la aplicación de estos filtros, la cantidad de votantes únicos se reduce a 48,119.**

In [11]:
# Leer metadatos de usuarios desde un archivo CSV
userMetadata = pd.read_csv(f'../data/Dataset 4, popup_screen_france.csv')
df = pd.merge(userMetadata[['user_id','politica']],df,on='user_id')
#Se filtran aquellos clientes que no seleccionaron partido politico

# Filtra el DataFrame para excluir las filas con política igual a 3 y 99
df_filtrado = df[df['politica'] != 3]
df_filtrado = df_filtrado[df_filtrado['politica'] != 99]
# Cargar el archivo TSV de etiquetas de coordenadas
labels_coords = pd.read_csv('../data/labels/Dataset 5, set_proposals_france.csv')
labels_coords = labels_coords.rename(columns={'issue_id':'id'})

# Ranking Trueskill

**Se implementa el algoritmo de trueskill para generar un ranking de las propuestas**

In [12]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))
from Libreria.ranking.trueskill import trueskill
from tqdm import tqdm
def trueskill_bootstrapping(df, iterations=50, sample=20000):
    options = []
    for i in tqdm(range(iterations)):
        df_temp = df.sample(sample).copy()
        options.append(trueskill(df_temp))
        pass

    options = pd.concat(options)
    
    return options

**Se aplicará trueskill para los usuarios de izquierda**

In [13]:
izquierda = df[df['politica']<3]
options_izquierda = trueskill_bootstrapping(izquierda, 100,sample=len(izquierda))

100%|██████████| 100/100 [6:36:58<00:00, 238.18s/it] 


In [14]:
ranking_bootstrapping_izquierda = options_izquierda.groupby("id").mean().reset_index()\
    .merge(labels_coords, on="id", how="left").sort_values("skill", ascending=False)

ranking_bootstrapping_izquierda["rank_2"] = range(1, ranking_bootstrapping_izquierda.shape[0] + 1)

**Se aplicará trueskill para los usuarios de derecha**

In [15]:
derecha = df[df['politica']>3]
options_derecha = trueskill_bootstrapping(derecha, 100,sample=len(derecha))

100%|██████████| 100/100 [3:10:13<00:00, 114.13s/it] 


In [16]:
ranking_bootstrapping_derecha = options_derecha.groupby("id").mean().reset_index()\
    .merge(labels_coords, on="id", how="left").sort_values("skill", ascending=False)

ranking_bootstrapping_derecha["rank_2"] = range(1, ranking_bootstrapping_derecha.shape[0] + 1)

**Implementación ratio**

In [17]:
ranking_bootstrapping_izquierda = ranking_bootstrapping_izquierda.rename(columns={'rank_2':'Ranking Izquierda','skill':'skill iz','rank':'rank iz'})
ranking_bootstrapping_derecha = ranking_bootstrapping_derecha.rename(columns={'rank_2':'Ranking Derecha','skill':'skill der','rank':'rank der'})
total = ranking_bootstrapping_izquierda[['id','Ranking Izquierda','es','es_category','skill iz','rank iz']].merge(ranking_bootstrapping_derecha[['id','Ranking Derecha','skill der','rank der']],on='id')
total['ranking_derecha_sort'] =total['Ranking Derecha']/total['Ranking Izquierda']
total['ranking_izquierda_sort'] =total['Ranking Izquierda']/total['Ranking Derecha']

total.head()

,id,Ranking Izquierda,es,es_category,skill iz,rank iz,Ranking Derecha,skill der,rank der,ranking_derecha_sort,ranking_izquierda_sort
0,24,1,Crear trabajos en los hospitales públicos,Políticas Públicas,68.104530,1.26,4,63.842983,4.57,4.000000,0.250000
1,97,2,Dedicar el 3% del PIB a la investigación y el ...,Políticas Públicas,66.947360,2.52,1,65.280621,1.67,0.500000,2.000000
2,23,3,Aumentar el número de médicos en áreas rurales...,Políticas Públicas,66.349800,4.03,2,65.105965,1.86,0.666667,1.500000
3,41,4,"Prohibir pesticidas peligrosos (por ejemplo, n...",Medio Ambiente y Energía,65.682763,6.72,3,64.053090,4.06,0.750000,1.333333
4,40,5,Desarrollar un impuesto que desalente la obsol...,Medio Ambiente y Energía,65.597725,7.22,5,63.470082,5.80,1.000000,1.000000


In [18]:
total['ranking_derecha_sort_skill'] =total['skill der']/total['skill iz']
total['ranking_izquierda_sort_skill'] =total['skill iz']/total['skill der']

In [19]:
total['ranking_derecha_sort_rank'] =total['rank der']/total['rank iz']
total['ranking_izquierda_sort_rank'] =total['rank iz']/total['rank der']

In [20]:
total.to_excel('results/Ranking_propuestas_trueskill_Francia.xlsx')

# Ranking Winrate

In [21]:
df["selected"] = df["option_a"].astype(int)
df["option_a"] = df["option_a"].astype(int)
df["option_b"] = df["option_b"].astype(int)
df["id"] = range(1, df.shape[0] + 1)

df["option_a_sorted"] = df[["option_a", "option_b"]].min(axis=1).astype(int)
df["option_b_sorted"] = df[["option_a", "option_b"]].max(axis=1).astype(int)
df["card_id"] = df["option_a_sorted"].astype(str) + "-" + df["option_b_sorted"].astype(str)

a = df[["option_a", "option_b", "selected"]].values
df["option_source"] = np.where(a[:, 1] == a[:, 2], a[:, 0], a[:, 1])
df["option_target"] = np.where(a[:, 0] == a[:, 2], a[:, 0], a[:, 1])

izquierda = df[df['politica']<5]
derecha = df[df['politica']>5]

def win_rate(df):
    dd = df.groupby(["option_source", "option_target"]).agg({"user_id": "count"}).reset_index()
    m = dd.pivot(index="option_source", columns="option_target", values="user_id").fillna(0)
    ids = set(df["option_source"]) | set(df["option_target"])
    m = m.reindex(ids)
    m = m.reindex(ids, axis=1)
    m = m.fillna(0)

    r = m + m.T
    win_rate = m.sum() / r.sum()

    return pd.DataFrame(win_rate).reset_index().rename(columns={"option_target": "id", 0: "agreement"})

In [22]:
izquierda_winrate = win_rate(izquierda)
derecha_winrate = win_rate(derecha)

In [23]:
izquierda_winrate = izquierda_winrate.merge(labels_coords, on="id").sort_values('agreement',ascending=False)
izquierda_winrate['ranking_politica'] = 'izquierda'
derecha_winrate = derecha_winrate.merge(labels_coords, on="id").sort_values('agreement',ascending=False)
derecha_winrate['ranking_politica'] = 'derecha'
winrate=pd.concat([izquierda_winrate,derecha_winrate])
winrate.head()

,id,agreement,multichoice,fr_category,fr,fr_multichoice,en_category,en,en_multichoice,es_category,es,es_multichoice,length,ranking_politica
23,24,0.938328,1,Politiques Publiques,Créer des emplois dans les hôpitaux publics,"['25 000', '100 000', '250 000']",Public policies,Increase personnel in public hospitals,"['25k', '100k', '250k']",Políticas Públicas,Crear trabajos en los hospitales públicos,"['25 mil', '100 mil', '250 mil']",43,izquierda
96,97,0.933537,0,Politiques Publiques,Consacrer 3 % du PIB à la recherche et au déve...,NaN,Public policies,Devote 3% of GDP to research and development,NaN,Políticas Públicas,Dedicar el 3% del PIB a la investigación y el ...,NaN,55,izquierda
40,41,0.909094,0,Environnement et Energie,Interdire les pesticides jugés dangereux par l...,NaN,Environment and Energy,Ban dangerous pesticides (eg neonicotinoides),NaN,Medio Ambiente y Energía,"Prohibir pesticidas peligrosos (por ejemplo, n...",NaN,91,izquierda
22,23,0.907836,0,Politiques Publiques,Combler les déserts médicaux,NaN,Public policies,Increase number of doctors in rural underserve...,NaN,Políticas Públicas,Aumentar el número de médicos en áreas rurales...,NaN,28,izquierda
39,40,0.897709,0,Environnement et Energie,Développer une fiscalité décourageant l'obsole...,NaN,Environment and Energy,Develop a taxation to discourage programmed ob...,NaN,Medio Ambiente y Energía,Desarrollar un impuesto que desalente la obsol...,NaN,63,izquierda
